# Beyond Discounts: Data-Driven Black Friday Sales Insights
**Data Mining – Year 1 Summative Assessment | Scenario 1**

**Role:** Data Analyst at InsightMart Analytics  
**Dataset:** Black Friday retail transaction data (537,577 rows)

---
## Stage 1: Define the Project Scope

### Objectives
1. **Understand shopping preferences** – How do demographics (age, gender, city) influence purchase amounts?
2. **Segment customers** – Use clustering to identify distinct buyer groups.
3. **Find cross-selling opportunities** – Apply association rule mining on product categories.
4. **Detect anomalies** – Spot unusually high spenders using statistical methods.
5. **Deploy insights** – Build an interactive Streamlit dashboard.

### Dataset Columns
| Column | Description |
|--------|-------------|
| User_ID | Unique customer ID |
| Product_ID | Unique product ID |
| Gender | M / F |
| Age | Age group (0-17, 18-25, 26-35, 36-45, 46-50, 51-55, 55+) |
| Occupation | Occupation code (0-20) |
| City_Category | City tier (A, B, C) |
| Stay_In_Current_City_Years | Years in city |
| Marital_Status | 0 = Single, 1 = Married |
| Product_Category_1/2/3 | Product category codes |
| Purchase | Purchase amount (USD) |

---
## Stage 2: Data Cleaning & Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

# Load dataset
df = pd.read_csv('BlackFriday.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# Check missing values
print('Missing values:')
print(df.isnull().sum())
print(f'\nTotal rows: {len(df):,}')

In [ ]:
# Handle missing values in Product_Category_2 and Product_Category_3
df['Product_Category_2'] = df['Product_Category_2'].fillna(0).astype(int)
df['Product_Category_3'] = df['Product_Category_3'].fillna(0).astype(int)

# Check and remove duplicates
print(f'Duplicate rows: {df.duplicated().sum()}')
df.drop_duplicates(inplace=True)
print(f'Shape after dedup: {df.shape}')

In [ ]:
# Encode Gender: M=0, F=1
df['Gender_Encoded'] = df['Gender'].map({'M': 0, 'F': 1})

# Encode Age groups into ordered numbers
age_order = {'0-17': 1, '18-25': 2, '26-35': 3, '36-45': 4,
             '46-50': 5, '51-55': 6, '55+': 7}
df['Age_Encoded'] = df['Age'].map(age_order)

# Encode City_Category
df['City_Encoded'] = df['City_Category'].map({'A': 1, 'B': 2, 'C': 3})

# Encode Stay_In_Current_City_Years
df['Stay_Encoded'] = df['Stay_In_Current_City_Years'].replace('4+', '4').astype(int)

print('Encoding complete.')
df[['Gender','Gender_Encoded','Age','Age_Encoded']].drop_duplicates().sort_values('Age_Encoded')

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df['Purchase_Normalized'] = scaler.fit_transform(df[['Purchase']])

print('Purchase stats (original):')
print(df['Purchase'].describe())
print('\nPurchase stats (normalized):')
print(df['Purchase_Normalized'].describe())

---
## Stage 3: Exploratory Data Analysis (EDA)

In [ ]:
# 3.1 Avg Purchase by Age Group and Purchase by Gender
age_order_list = ['0-17','18-25','26-35','36-45','46-50','51-55','55+']
age_means = df.groupby('Age')['Purchase'].mean().reindex(age_order_list)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(age_means.index, age_means.values, color='steelblue', edgecolor='white')
axes[0].set_title('Avg Purchase by Age Group', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Avg Purchase (USD)')
axes[0].tick_params(axis='x', rotation=20)

df.boxplot(column='Purchase', by='Gender', ax=axes[1],
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Purchase Distribution by Gender', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Purchase (USD)')
plt.suptitle('')
plt.tight_layout()
plt.savefig('eda_age_gender.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3.2 Most popular product categories
cat1_counts = df['Product_Category_1'].value_counts().head(15)

plt.figure(figsize=(12, 5))
plt.bar(cat1_counts.index.astype(str), cat1_counts.values, color='coral', edgecolor='white')
plt.title('Top 15 Product Categories (Category 1)', fontsize=13, fontweight='bold')
plt.xlabel('Product Category')
plt.ylabel('Number of Transactions')
plt.tight_layout()
plt.savefig('eda_categories.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3.3 Scatter: Avg Purchase vs Occupation
occ_means = df.groupby('Occupation')['Purchase'].mean().reset_index()

plt.figure(figsize=(12, 5))
plt.scatter(occ_means['Occupation'], occ_means['Purchase'],
            color='mediumpurple', s=80, edgecolors='white', zorder=3)
for _, row in occ_means.iterrows():
    plt.annotate(f"{row['Purchase']:.0f}",
                 (row['Occupation'], row['Purchase']),
                 textcoords='offset points', xytext=(0, 6), fontsize=7, ha='center')
plt.title('Avg Purchase by Occupation Code', fontsize=13, fontweight='bold')
plt.xlabel('Occupation Code')
plt.ylabel('Avg Purchase (USD)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('eda_occupation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3.4 Average purchase per product category
cat_avg = df.groupby('Product_Category_1')['Purchase'].mean().sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 5))
plt.bar(cat_avg.index.astype(str), cat_avg.values, color='teal', edgecolor='white')
plt.title('Avg Purchase Amount by Product Category', fontsize=13, fontweight='bold')
plt.xlabel('Product Category')
plt.ylabel('Avg Purchase (USD)')
plt.tight_layout()
plt.savefig('eda_avg_purchase.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3.5 Correlation heatmap
corr_cols = ['Age_Encoded', 'Occupation', 'City_Encoded', 'Stay_Encoded',
             'Marital_Status', 'Product_Category_1', 'Purchase']

plt.figure(figsize=(9, 7))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Stage 4: Clustering Analysis

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Aggregate per user
user_df = df.groupby('User_ID').agg(
    Total_Purchase=('Purchase', 'sum'),
    Avg_Purchase=('Purchase', 'mean'),
    Num_Transactions=('Purchase', 'count'),
    Age_Encoded=('Age_Encoded', 'first'),
    Occupation=('Occupation', 'first'),
    Marital_Status=('Marital_Status', 'first'),
    Gender_Encoded=('Gender_Encoded', 'first'),
    City_Encoded=('City_Encoded', 'first')
).reset_index()

print(f'Unique customers: {len(user_df):,}')
user_df.head()

In [ ]:
# Elbow Method
cluster_features = ['Total_Purchase', 'Avg_Purchase', 'Num_Transactions',
                    'Age_Encoded', 'Occupation']

X = user_df[cluster_features]
scaler_c = StandardScaler()
X_scaled = scaler_c.fit_transform(X)

inertias = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(range(2, 11), inertias, marker='o', color='steelblue', linewidth=2)
plt.title('Elbow Method – Optimal k', fontsize=13, fontweight='bold')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('clustering_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Apply K-Means with k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
user_df['Cluster'] = kmeans.fit_predict(X_scaled)

means = user_df.groupby('Cluster')['Total_Purchase'].mean().sort_values()
label_map = {
    means.index[0]: 'Budget Shoppers',
    means.index[1]: 'Casual Buyers',
    means.index[2]: 'Regular Spenders',
    means.index[3]: 'Premium Buyers'
}
user_df['Cluster_Label'] = user_df['Cluster'].map(label_map)

print('Cluster distribution:')
print(user_df['Cluster_Label'].value_counts())

In [ ]:
# Cluster scatter plot
COLORS = {'Budget Shoppers': '#FF9800', 'Casual Buyers': '#2196F3',
          'Regular Spenders': '#9C27B0', 'Premium Buyers': '#4CAF50'}

plt.figure(figsize=(10, 6))
for label, color in COLORS.items():
    mask = user_df['Cluster_Label'] == label
    plt.scatter(user_df.loc[mask, 'Num_Transactions'],
                user_df.loc[mask, 'Total_Purchase'],
                label=label, alpha=0.5, s=30, color=color)
plt.title('Customer Clusters: Transactions vs Total Spend', fontsize=13, fontweight='bold')
plt.xlabel('Number of Transactions')
plt.ylabel('Total Purchase (USD)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('clustering_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cluster summary statistics
summary = user_df.groupby('Cluster_Label')[cluster_features].mean().round(2)
print('Cluster Summary (mean values):')
print(summary)

---
## Stage 5: Association Rule Mining

In [ ]:
# pip install mlxtend  (run this if needed)
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

def make_transaction(row):
    cats = [f"Cat_{int(row['Product_Category_1'])}"]
    if row['Product_Category_2'] != 0:
        cats.append(f"Cat_{int(row['Product_Category_2'])}")
    if row['Product_Category_3'] != 0:
        cats.append(f"Cat_{int(row['Product_Category_3'])}")
    return list(set(cats))

transactions = df.apply(make_transaction, axis=1).tolist()

te = TransactionEncoder()
te_array = te.fit_transform(transactions)
te_df = pd.DataFrame(te_array, columns=te.columns_)

print(f'Transaction matrix: {te_df.shape}')

frequent_itemsets = apriori(te_df, min_support=0.05, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
print(f'Frequent itemsets: {len(frequent_itemsets)}')
frequent_itemsets.sort_values('support', ascending=False).head(10)

In [ ]:
# Generate association rules
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.4)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f'Rules generated: {len(rules)}')
rules[['antecedents','consequents','support','confidence','lift']].head(10)

In [ ]:
# Visualise top rules
top_rules = rules.head(10).copy()
top_rules['rule'] = top_rules.apply(
    lambda r: f"{', '.join(list(r['antecedents']))}  ->  {', '.join(list(r['consequents']))}", axis=1)

plt.figure(figsize=(12, 5))
plt.barh(top_rules['rule'], top_rules['lift'], color='coral', edgecolor='white')
plt.xlabel('Lift')
plt.title('Top 10 Association Rules by Lift', fontsize=13, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('association_rules.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Stage 6: Anomaly Detection

In [ ]:
from scipy import stats

# Total spend per user
user_spend = df.groupby('User_ID')['Purchase'].sum().reset_index()
user_spend.columns = ['User_ID', 'Total_Spend']

# Z-Score
user_spend['ZScore'] = np.abs(stats.zscore(user_spend['Total_Spend']))
anomalies_z = user_spend[user_spend['ZScore'] > 3]
print(f'Z-Score anomalies (|z| > 3): {len(anomalies_z)} customers')

# IQR
Q1 = user_spend['Total_Spend'].quantile(0.25)
Q3 = user_spend['Total_Spend'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR
lower = Q1 - 1.5 * IQR
anomalies_iqr = user_spend[(user_spend['Total_Spend'] > upper) | (user_spend['Total_Spend'] < lower)]
print(f'IQR anomalies: {len(anomalies_iqr)} customers')
print(f'IQR upper bound: ${upper:,.0f}')

In [ ]:
# Visualise anomalies
plt.figure(figsize=(12, 5))
normal = user_spend[user_spend['ZScore'] <= 3]
anomaly = user_spend[user_spend['ZScore'] > 3]

plt.scatter(range(len(normal)), normal['Total_Spend'],
            color='steelblue', alpha=0.3, s=15, label='Normal Customers')
plt.scatter(anomaly.index, anomaly['Total_Spend'],
            color='red', s=60, zorder=5, label=f'Anomalies ({len(anomaly)})')
plt.axhline(upper, color='orange', linestyle='--', label=f'IQR Upper (${upper:,.0f})')
plt.title('Anomaly Detection – Customer Total Spend', fontsize=13, fontweight='bold')
plt.xlabel('Customer Index')
plt.ylabel('Total Spend (USD)')
plt.legend()
plt.tight_layout()
plt.savefig('anomaly_detection.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Profile anomalous customers
anomaly_profiles = anomalies_z.merge(
    df[['User_ID','Age','Gender','Occupation','City_Category']].drop_duplicates('User_ID'),
    on='User_ID'
)
print('Top anomalous customers:')
anomaly_profiles.sort_values('Total_Spend', ascending=False).head(10)

---
## Stage 7: Insights & Reporting

In [ ]:
print('=' * 65)
print('KEY INSIGHTS - Black Friday Sales Analysis')
print('=' * 65)

top_age = df.groupby('Age')['Purchase'].mean().idxmax()
top_age_val = df.groupby('Age')['Purchase'].mean().max()
print(f'\n1. Highest spending age group: {top_age} (avg ${top_age_val:,.0f}/transaction)')

gender_spend = df.groupby('Gender')['Purchase'].mean()
print(f'2. Avg purchase - Male: ${gender_spend["M"]:,.0f} | Female: ${gender_spend["F"]:,.0f}')

top_cat = df['Product_Category_1'].value_counts().idxmax()
print(f'3. Most purchased product category: {top_cat}')

city_spend = df.groupby('City_Category')['Purchase'].mean()
print(f'4. City B avg: ${city_spend["B"]:,.0f} | City C: ${city_spend["C"]:,.0f} | City A: ${city_spend["A"]:,.0f}')

print(f'5. Anomalous high-spenders: {len(anomalies_z)} customers (z-score > 3)')

print('6. Customer segments:')
print(user_df['Cluster_Label'].value_counts().to_string())
print('=' * 65)

In [ ]:
# Save cleaned datasets for Streamlit app
df.to_csv('BlackFriday_cleaned.csv', index=False)
user_df.to_csv('BlackFriday_users.csv', index=False)
print('Saved: BlackFriday_cleaned.csv and BlackFriday_users.csv')

---
## Stage 8: Deployment on Streamlit Cloud

See `app.py` and `requirements.txt` in this repository.

### Steps to deploy:
1. Push this repo to GitHub: `IDAI105[YourStudentID]-YourName`
2. Go to [https://streamlit.io/cloud](https://streamlit.io/cloud)
3. Sign in with GitHub → New App → Select repo → Select `app.py`
4. Click **Deploy** → Copy the live URL for your submission PDF